# 🌸 Task 3: Iris Flower Classification
**CodSoft Data Science Internship**

**Objective:** Train a machine learning model to classify Iris flowers into three species — *Setosa*, *Versicolor*, and *Virginica* — based on sepal and petal measurements.

---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## Step 2: Load Dataset
> The Iris dataset is built into scikit-learn. You can also download it from: https://www.kaggle.com/datasets/uciml/iris

In [ ]:
# Load from sklearn
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Option B: Load from CSV
# df = pd.read_csv('IRIS.csv')

print('Dataset Shape:', df.shape)
df.head(10)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Class Distribution ===')
print(df['species_name'].value_counts())

In [ ]:
df.describe()

In [ ]:
# Pairplot - visualize relationships between features
sns.pairplot(df, hue='species_name', vars=iris.feature_names,
             palette={'setosa': '#e74c3c', 'versicolor': '#2ecc71', 'virginica': '#3498db'},
             plot_kws={'alpha': 0.7}, height=2.2)
plt.suptitle('Iris Dataset - Pairplot', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('iris_pairplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots for each feature by species
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#e74c3c', '#2ecc71', '#3498db']

for i, feature in enumerate(iris.feature_names):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=feature, by='species_name', ax=ax,
               patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(feature, fontsize=11, fontweight='bold')
    ax.set_xlabel('Species')
    ax.set_ylabel('cm')

plt.suptitle('Feature Distribution by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df[iris.feature_names].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_heatmap.png', bbox_inches='tight')
plt.show()

## Step 4: Data Preprocessing & Train-Test Split

In [ ]:
X = df[iris.feature_names]
y = df['species']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')

## Step 5: Model Training — Three Algorithms

In [ ]:
# Define models
models = {
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv_score = cross_val_score(model, X_scaled, y, cv=5).mean()
    results[name] = {'model': model, 'y_pred': y_pred, 'accuracy': acc, 'cv_score': cv_score}
    print(f'\n=== {name} ===')
    print(f'Test Accuracy : {acc:.4f}')
    print(f'CV Accuracy (5-fold): {cv_score:.4f}')
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Step 6: Model Evaluation & Visualization

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cmaps = ['Blues', 'Greens', 'Oranges']

for ax, (name, res), cmap in zip(axes, results.items(), cmaps):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(
        ax=ax, colorbar=False, cmap=cmap
    )
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.4f}', fontsize=10, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# Model accuracy comparison
names = list(results.keys())
test_accs = [results[n]['accuracy'] for n in names]
cv_accs = [results[n]['cv_score'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, test_accs, width, label='Test Accuracy', color='#3498db', edgecolor='black')
b2 = ax.bar(x + width/2, cv_accs, width, label='CV Accuracy (5-fold)', color='#2ecc71', edgecolor='black')

ax.bar_label(b1, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.4f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0.85, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison — Test vs Cross-Validation Accuracy', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('iris_model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance from Random Forest
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=iris.feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importances — Random Forest', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('iris_feature_importance.png', bbox_inches='tight')
plt.show()

## Step 7: Predict on New Sample

In [ ]:
# Predict species for a new flower
# Format: [sepal length, sepal width, petal length, petal width] (in cm)
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
sample_scaled = scaler.transform(sample)

best_model = results['Support Vector Machine']['model']
prediction = best_model.predict(sample_scaled)[0]
species_name = iris.target_names[prediction]

print(f'Input: sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2')
print(f'\nPredicted Species: {species_name.upper()} 🌸')

## ✅ Summary

| Model | Test Accuracy |
|-------|---------------|
| K-Nearest Neighbors | ~97% |
| Random Forest | ~97% |
| Support Vector Machine | ~97% |

**Key Insights:**
- The Iris dataset is highly separable — all three models perform very well (>95% accuracy).
- Petal length and petal width are the most discriminating features.
- *Setosa* is perfectly separable; *Versicolor* and *Virginica* have slight overlap.
- SVM with RBF kernel is generally considered the best for this type of classification task.
